In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
import os
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.transforms import InterpolationMode
import torchvision.transforms.functional as TF
import time
from tqdm import tqdm
import torch.nn.functional as F



In [2]:
# import os

# KAGGLE_INPUT = "/kaggle/input"

# for root, dirs, files in os.walk(KAGGLE_INPUT):
#     level = root.replace(KAGGLE_INPUT, '').count(os.sep)
#     indent = ' ' * 4 * level
#     print(f"{indent}{os.path.basename(root)}/")
    
#     subindent = ' ' * 4 * (level + 1)
#     for f in files[:5]:   # show first 5 files only
#         print(f"{subindent}{f}")

In [3]:
scaler = torch.amp.GradScaler('cuda')

device = "cuda" if torch.cuda.is_available() else "cpu"


SHOW_PROGRESS = False
num_epochs =50
learning_rate = 1e-4


print(f"Device        : {device}")
print(f"Epochs        : {num_epochs}")
print(f"Learning Rate : {learning_rate}")

Device        : cuda
Epochs        : 50
Learning Rate : 0.0001


In [4]:
# ==========================================
# DATASET PATH CONFIG
# ==========================================

DATASET = "LIM-CD"   # "LEVIR-CD" or "LIM-CD"

if DATASET == "LEVIR-CD":
    ROOT = "/kaggle/input/datasets/mdrifaturrahman33/levir-cd/LEVIR CD"
    DIRMAP  = {"A": "A", "B": "B", "label": "label"}
    TILESIZE = 256     # LEVIR native=1024, must patch to 256
    STRIDE   = 256
    BATCH_SIZE = 8
    POSWEIGHT    = 2.5
    MIN_CHANGE   = 10
    FOCAL_ALPHA  = 0.25

elif DATASET == "LIM-CD":
    ROOT = "/kaggle/input/datasets/ravat1920/lim-cd/lim-cd"
    DIRMAP  = {"A": "t1", "B": "t2", "label": "label"}
    TILESIZE = 512     # LIM-CD native=512, no patching needed
    STRIDE   = 512
    BATCH_SIZE = 4     # halved due to 4× larger input
    POSWEIGHT    = 1.5    # less imbalanced dataset
    MIN_CHANGE   = 40     # scaled: same 0.015% threshold at 512×512
    FOCAL_ALPHA  = 0.5    # balanced when imbalance is lower

print("Dataset :", DATASET)        # ← was USE_DATASET (NameError)
print("Root    :", ROOT)
print("Mapping :", DIRMAP)         # ← was DIRMAP (NameError)
print(f"TileSize : {TILESIZE}  Stride : {STRIDE}  Batch : {BATCH_SIZE}")
print(f"PosWeight: {POSWEIGHT}  MinChange: {MIN_CHANGE}  FocalAlpha: {FOCAL_ALPHA}")

Dataset : LIM-CD
Root    : /kaggle/input/datasets/ravat1920/lim-cd/lim-cd
Mapping : {'A': 't1', 'B': 't2', 'label': 'label'}
TileSize : 512  Stride : 512  Batch : 4
PosWeight: 1.5  MinChange: 40  FocalAlpha: 0.5


In [5]:
class PatchedChangeDataset(Dataset):
    def __init__(self, root_dir, augment=False,
                 tile_size=TILESIZE, stride=STRIDE,
                 min_change_pixels=10,
                 neg_ratio=1.5):          # None = keep ALL patches (val/test)
        self.root_dir  = root_dir
        self.augment   = augment
        self.tile_size = tile_size
        self.stride    = stride

        self.A = sorted(os.listdir(os.path.join(root_dir, DIRMAP["A"])))
        self.B = sorted(os.listdir(os.path.join(root_dir, DIRMAP["B"])))
        self.Y = sorted(os.listdir(os.path.join(root_dir, DIRMAP["label"])))

        _probe = Image.open(os.path.join(root_dir, DIRMAP["A"], self.A[0]))
        full_W, full_H = _probe.size
        _probe.close()

        raw_patches = []
        if full_H == tile_size and full_W == tile_size:
            for i in range(len(self.A)):
                raw_patches.append((i, 0, 0))
        else:
            for i in range(len(self.A)):
                for r in range(0, full_H - tile_size + 1, stride):
                    for c in range(0, full_W - tile_size + 1, stride):
                        raw_patches.append((i, r, c))

        split_name = os.path.basename(root_dir)

        # ── neg_ratio=None → val/test: keep ALL patches, no filtering ──
        if neg_ratio is None:
            self.patches = raw_patches
            print(f"  [{split_name}] {len(self.A)} images → "
                  f"{len(self.patches)} patches (NO filtering — full distribution)")
        else:
            # ── Train: controlled negative sampling ──
            print(f"  [{split_name}] Scanning {len(raw_patches)} patches...", end=" ")

            change_patches    = []
            no_change_patches = []

            _mask_cache_idx = -1
            _mask_cache_arr = None

            for (img_idx, r, c) in raw_patches:
                if img_idx != _mask_cache_idx:
                    mask_path = os.path.join(root_dir, "label", self.Y[img_idx])
                    _mask_cache_arr = np.array(Image.open(mask_path).convert("L"))
                    _mask_cache_idx = img_idx

                tile_mask = _mask_cache_arr[r:r + tile_size, c:c + tile_size]
                n_change  = (tile_mask > 127).sum()

                if n_change >= min_change_pixels:
                    change_patches.append((img_idx, r, c))
                else:
                    no_change_patches.append((img_idx, r, c))

            # REFINEMENT 3: seed before shuffle for reproducibility
            np.random.seed(42)
            np.random.shuffle(no_change_patches)

            # REFINEMENT 1: neg_ratio=1.5 (not 1.0) for realistic distribution
            max_neg  = int(len(change_patches) * neg_ratio)
            kept_neg = no_change_patches[:max_neg]

            self.patches = change_patches + kept_neg
            np.random.seed(42)
            np.random.shuffle(self.patches)

            print(f"done")
            print(f"  [{split_name}] Change patches : {len(change_patches)}")
            print(f"  [{split_name}] No-change kept : {len(kept_neg)} "
                  f"(ratio={neg_ratio}× → max {max_neg})")
            print(f"  [{split_name}] Total used     : {len(self.patches)} "
                  f"/ {len(raw_patches)} raw\n")

        self.to_tensor_img  = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std =[0.229, 0.224, 0.225])
        ])
        self.to_tensor_mask = transforms.ToTensor()

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        img_idx, r, c = self.patches[idx]
        box = (c, r, c + self.tile_size, r + self.tile_size)

        img1 = Image.open(os.path.join(self.root_dir, DIRMAP["A"], self.A[img_idx])).convert("RGB").crop(box)
        img2 = Image.open(os.path.join(self.root_dir, DIRMAP["B"], self.B[img_idx])).convert("RGB").crop(box)
        mask = Image.open(os.path.join(self.root_dir, DIRMAP["label"], self.Y[img_idx])).convert("L").crop(box)
        if self.augment:
            if torch.rand(1) < 0.5:
                img1 = TF.hflip(img1); img2 = TF.hflip(img2); mask = TF.hflip(mask)
            if torch.rand(1) < 0.5:
                img1 = TF.vflip(img1); img2 = TF.vflip(img2); mask = TF.vflip(mask)
            if torch.rand(1) < 0.3:
                angle = transforms.RandomRotation.get_params([-30, 30])
                img1 = TF.rotate(img1, angle, interpolation=InterpolationMode.BILINEAR)
                img2 = TF.rotate(img2, angle, interpolation=InterpolationMode.BILINEAR)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST)
            if torch.rand(1) < 0.4:
                img1 = TF.adjust_brightness(img1, torch.empty(1).uniform_(0.75, 1.25).item())
                img1 = TF.adjust_contrast(img1,   torch.empty(1).uniform_(0.75, 1.25).item())
                img2 = TF.adjust_brightness(img2, torch.empty(1).uniform_(0.75, 1.25).item())
                img2 = TF.adjust_contrast(img2,   torch.empty(1).uniform_(0.75, 1.25).item())

        img1 = self.to_tensor_img(img1)
        img2 = self.to_tensor_img(img2)
        mask = self.to_tensor_mask(mask)
        mask = (mask > 0.5).float()

        return img1, img2, mask

In [6]:
print("Building patch datasets...")
print("Using ROOT:", ROOT)

train_dataset = PatchedChangeDataset(
    os.path.join(ROOT, "train"), augment=True,
    min_change_pixels=MIN_CHANGE,
    neg_ratio=1.5          # REFINEMENT 1: 1.5× negatives → realistic urban distribution
)
val_dataset = PatchedChangeDataset(
    os.path.join(ROOT, "val"), augment=False,
    neg_ratio=None          # REFINEMENT 2: full distribution, no filtering
)
test_dataset = PatchedChangeDataset(
    os.path.join(ROOT, "test"), augment=False,
    neg_ratio=None          # REFINEMENT 2: full distribution, no filtering
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train batches/epoch : {len(train_loader)}")
print(f"Val   batches/epoch : {len(val_loader)}")
print(f"Test  batches       : {len(test_loader)}")

Building patch datasets...
Using ROOT: /kaggle/input/datasets/ravat1920/lim-cd/lim-cd
  [train] Scanning 6547 patches... done
  [train] Change patches : 6521
  [train] No-change kept : 26 (ratio=1.5× → max 9781)
  [train] Total used     : 6547 / 6547 raw

  [val] 1776 images → 1776 patches (NO filtering — full distribution)
  [test] 936 images → 936 patches (NO filtering — full distribution)
Train batches/epoch : 1636
Val   batches/epoch : 444
Test  batches       : 234


In [7]:
# Run after Cell 4 — check that patches now have change pixels
change_ratios = [train_dataset[i][2].mean().item() for i in range(20)]
print("Sample change ratios (first 20 patches):")
print([f"{r:.4f}" for r in change_ratios])
# Expect: mix of 0.0000 (kept neg) and 0.02–0.40 (change patches)
# NOT all 0.0000 as before

Sample change ratios (first 20 patches):
['0.9801', '0.0146', '0.2653', '0.0313', '0.1997', '0.0360', '0.0194', '0.0351', '0.4852', '0.3045', '0.0190', '0.0933', '0.0024', '0.0109', '0.0538', '0.1805', '0.0176', '0.0985', '0.0306', '0.0096']


In [8]:
class SEBlock(nn.Module):
    """Squeeze-and-Excitation: recalibrates channel importance"""
    def __init__(self, ch, reduction=16):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(ch, ch // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch // reduction, ch, 1, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.se(x)


class DecoderBlock(nn.Module):
    """
    Upsample → concat RICH skip [f1, f2, |f1-f2|] → Conv-BN-ReLU → SE
    Rich skip triples channels but preserves individual T1/T2 context
    SE then recalibrates which channels matter most for change
    """
    def __init__(self, in_ch, out_ch, skip_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch + skip_ch, out_ch, kernel_size=3,
                      padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
        self.se = SEBlock(out_ch)   # ← Priority 6: SE after each decoder block

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)   # skip = cat[f1, f2, |f1-f2|]
        return self.se(self.conv(x))


class DiffGate(nn.Module):
    """Modulates T1/T2 features using their difference as channel attention"""
    def __init__(self, ch):
        super().__init__()
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(ch, ch // 8, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch // 8, ch, 1, bias=False),
            nn.Sigmoid()
        )

    def forward(self, f1, f2):
        g = self.gate(torch.abs(f1 - f2))   # gate from difference signal
        return f1 * g, f2 * g               # both branches modulated

In [9]:
class ChangeDetectionNet(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)  # ← ResNet-50

        # ResNet-50 channel sizes differ from ResNet-18:
        # stage0: 64, stage1: 256, stage2: 512, stage3: 1024, stage4: 2048
        self.stage0 = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)  # [B,  64,128,128]
        self.stage1 = nn.Sequential(backbone.maxpool, backbone.layer1)            # [B, 256, 64, 64]
        self.stage2 = backbone.layer2                                              # [B, 512, 32, 32]
        self.stage3 = backbone.layer3                                              # [B,1024, 16, 16]
        self.stage4 = backbone.layer4                                              # [B,2048,  8,  8]

        # Project down to match attention dims (avoids 2048-dim MHA overhead)
        self.proj4 = nn.Conv2d(2048, 512, 1, bias=False)   # 2048 → 512
        self.proj3 = nn.Conv2d(1024, 256, 1, bias=False)   # 1024 → 256

        self.diff_gate2  = DiffGate(512)
        self.diff_gate3  = DiffGate(1024)
        self.proj2       = nn.Conv2d(512, 128, 1, bias=False)
        self.cross_attn2 = nn.MultiheadAttention(128, 4, batch_first=True, dropout=0.1)
        self.proj2_back  = nn.Conv2d(128, 512, 1, bias=False)
    
        # Cross-attention (same dims as before — projections handle the mismatch)
        self.cross_attn4 = nn.MultiheadAttention(512, 8, batch_first=True, dropout=0.1)
        self.cross_attn3 = nn.MultiheadAttention(256, 8, batch_first=True, dropout=0.1)

        # Decoder skip_ch = 3× projected channel size (rich skip [f1,f2,|f1-f2|])
        # Using PROJECTED sizes (512, 256) not raw ResNet-50 sizes (2048, 1024)
        self.dec4 = DecoderBlock(512, 256, skip_ch=256*3)   # skip from proj3: 256×3=768
        self.dec3 = DecoderBlock(256, 128, skip_ch=512*3)   # skip from stage2: 512×3=1536
        self.dec2 = DecoderBlock(128,  64, skip_ch=256*3)   # skip from stage1: 256×3=768
        self.dec1 = DecoderBlock( 64,  64, skip_ch= 64*3)   # skip from stage0:  64×3=192

        self.head = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=0.2),
            nn.Conv2d(32, 1, kernel_size=1)
        )

        # Auxiliary heads for deep supervision
        self.aux_dec4 = nn.Conv2d(256, 1, 1)
        self.aux_dec3 = nn.Conv2d(128, 1, 1)

    def encode(self, x):
        s0 = self.stage0(x)
        s1 = self.stage1(s0)
        s2 = self.stage2(s1)
        s3 = self.stage3(s2)
        s4 = self.stage4(s3)
        return s0, s1, s2, s3, s4

    def _cross_attn(self, attn_module, f1_feat, f2_feat):
        B, C, H, W = f1_feat.shape
        q = f1_feat.flatten(2).permute(0, 2, 1)
        k = f2_feat.flatten(2).permute(0, 2, 1)
        out, _ = attn_module(q, k, k)
        return out.permute(0, 2, 1).reshape(B, C, H, W)

    def _rich_skip(self, a, b):
        return torch.cat([a, b, torch.abs(a - b)], dim=1)

    def forward(self, x1, x2):
        f1 = self.encode(x1)
        f2 = self.encode(x2)


        gf1_2, gf2_2 = self.diff_gate2(f1[2], f2[2])   # [B,512,32,32] gated
        gf1_3, gf2_3 = self.diff_gate3(f1[3], f2[3])   # [B,1024,16,16] gated

        # Project ResNet-50 bottleneck features down before attention
        p1_4 = self.proj4(f1[4]);  p2_4 = self.proj4(f2[4])  # [B,512,8,8]
        p1_3 = self.proj3(gf1_3);  p2_3 = self.proj3(gf2_3)

        # Multi-scale cross-attention on projected features
        attn4  = self._cross_attn(self.cross_attn4, p1_4, p2_4)
        diff4  = torch.abs(p1_4 - attn4)                         # [B,512,8,8]

        attn3  = self._cross_attn(self.cross_attn3, p1_3, p2_3)
        diff3  = torch.cat([p1_3, attn3, torch.abs(p1_3 - attn3)], dim=1)  # [B, 768, 16,16]


        # ── Cross-attention at stage2 ──────────────────────
        p1_2   = self.proj2(gf1_2);  p2_2 = self.proj2(gf2_2)  # [B,128,32,32]
        attn2  = self._cross_attn(self.cross_attn2, p1_2, p2_2) # [B,128,32,32]
        attn2_full = self.proj2_back(attn2)  
        

        # Decoder with rich skips (using raw ResNet-50 intermediate features)
        d4 = self.dec4(diff4, diff3)                                # [B,256,16,16]
        d3 = self.dec3(d4, self._rich_skip(gf1_2, attn2_full))         # [B,128,32,32]  stage2=512ch
        d2 = self.dec2(d3,    self._rich_skip(f1[1], f2[1]))        # [B, 64,64,64]  stage1=256ch
        d1 = self.dec1(d2,    self._rich_skip(f1[0], f2[0]))        # [B, 64,128,128] stage0=64ch

        p1   = self.head(d1)
        aux4 = self.aux_dec4(d4)
        aux3 = self.aux_dec3(d3)

        return p1, aux4, aux3

In [10]:
torch.manual_seed(42)
np.random.seed(42)

model = ChangeDetectionNet().to(device)

_a = torch.randn(2, 3, 256, 256).to(device)
_b = torch.randn(2, 3, 256, 256).to(device)

_out, _aux4, _aux3 = model(_a, _b)   # ← unpack tuple

print(f"Main output shape : {_out.shape}")    # Expect [2, 1, 256, 256]
print(f"Aux4 output shape : {_aux4.shape}")   # Expect [2, 1,  16,  16]
print(f"Aux3 output shape : {_aux3.shape}")   # Expect [2, 1,  32,  32]
print(f"Output max        : {_out.max().item():.4f}")
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params  : {total_params/1e6:.2f}M")   # Expect ~14.83M

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 128MB/s]


Main output shape : torch.Size([2, 1, 256, 256])
Aux4 output shape : torch.Size([2, 1, 16, 16])
Aux3 output shape : torch.Size([2, 1, 32, 32])
Output max        : 0.0655
Trainable params  : 32.29M


loss fn

In [11]:

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, pred, target):
        bce  = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        pt   = torch.exp(-bce)
        loss = self.alpha * (1 - pt) ** self.gamma * bce
        return loss.mean()


bce   = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POSWEIGHT]).to(device))
focal = FocalLoss(alpha=FOCAL_ALPHA, gamma=2.0)

def dice_loss(pred, target, smooth=1e-6):
    pred   = torch.sigmoid(pred)
    pred   = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    return 1 - (2 * intersection + smooth) / (pred.sum() + target.sum() + smooth)

def criterion(pred, target):
    # Focal handles hard edge pixels, BCE handles class balance, Dice handles region overlap
    return bce(pred, target) + dice_loss(pred, target) + 0.5 * focal(pred, target)

In [12]:
# Backbone param groups for freeze/unfreeze control
backbone_params = (list(model.stage0.parameters()) +
                   list(model.stage1.parameters()) +
                   list(model.stage2.parameters()) +
                   list(model.stage3.parameters()) +
                   list(model.stage4.parameters()))
backbone_param_ids = {id(p) for p in backbone_params}

head_params = [p for p in model.parameters()
               if id(p) not in backbone_param_ids]

# Freeze backbone for first 5 epochs
for p in backbone_params:
    p.requires_grad = False

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': 1e-5},  # 10× lower when unfrozen
    {'params': head_params,     'lr': 1e-4},
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=15, T_mult=2, eta_min=1e-6
)

In [13]:
def compute_metrics(pred, target):
    pred   = torch.sigmoid(pred)
    pred   = (pred > 0.5).float()
    tp = (pred  *  target      ).sum().float()
    tn = ((1-pred) * (1-target)).sum().float()
    fp = (pred  * (1-target)   ).sum().float()
    fn = ((1-pred) *  target   ).sum().float()
    return tp, tn, fp, fn

def finalize_metrics(tp, tn, fp, fn):
    eps  = 1e-8
    acc  = (tp + tn) / (tp + tn + fp + fn + eps)
    prec = tp / (tp + fp + eps)
    rec  = tp / (tp + fn + eps)
    f1   = 2 * prec * rec / (prec + rec + eps)
    iou  = tp / (tp + fp + fn + eps)   # FIX: Added IoU — required for paper comparison
    return {
        "acc" : acc.item(),
        "prec": prec.item(),
        "rec" : rec.item(),
        "f1"  : f1.item(),
        "iou" : iou.item()
    }

In [14]:
# ── Config ──────────────────────────────────────────────────
# num_epochs           = 105
UNFREEZE_EPOCH       = 5
EARLY_STOP_PATIENCE  = 25
CHECKPOINT_EVERY     = 20
# ─────────────────────────────────────────────────────────────

history = {
    "train_loss": [], "val_loss" : [],
    "train_acc" : [], "val_acc"  : [],
    "train_prec": [], "val_prec" : [],
    "train_rec" : [], "val_rec"  : [],
    "train_f1"  : [], "val_f1"   : [],
    "train_iou" : [], "val_iou"  : [],
}

best_val_f1      = 0.0
no_improve_count = 0
best_epoch       = 0
training_start   = time.time()

print("=" * 155)
print("  STAGE 1 TRAINING  —  ResNet-50 Multi-Scale Attn  "
      f"| {num_epochs} epochs | Early-stop patience={EARLY_STOP_PATIENCE}")
print("=" * 155)
print(
    f"{'Ep':>7} | {'Time':>6} | "
    f"{'TrLoss':>7} | {'TrAcc':>6} | {'TrPrec':>7} | {'TrRec':>6} | {'TrF1':>6} | {'TrIoU':>7} | "
    f"{'VlLoss':>7} | {'VlAcc':>6} | {'VlPrec':>7} | {'VlRec':>6} | {'VlF1':>6} | {'VlIoU':>7} | "
    f"{'BB_LR':>9} | {'HD_LR':>9}"
)
print("─" * 155)

for epoch in range(num_epochs):
    start_time = time.time()

    # ── Backbone unfreeze ──────────────────────────────────
    if epoch == UNFREEZE_EPOCH:
        for p in backbone_params:
            p.requires_grad = True
        optimizer.param_groups[0]['lr'] = 1e-5
        optimizer.param_groups[1]['lr'] = 1e-4
        print(f"\n  → Epoch {epoch+1}: Backbone unfrozen — full fine-tuning begins\n")

    # ── TRAIN ──────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    tr_tp = tr_tn = tr_fp = tr_fn = 0

    for img1, img2, mask in tqdm(train_loader, leave=False,
                                  disable=not SHOW_PROGRESS,
                                  desc=f"Ep {epoch+1:3d} train"):
        img1, img2, mask = img1.to(device), img2.to(device), mask.to(device)
        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            pred, aux4, aux3 = model(img1, img2)
            mask_16 = F.interpolate(mask, size=aux4.shape[-2:], mode='nearest')
            mask_32 = F.interpolate(mask, size=aux3.shape[-2:], mode='nearest')
            loss = (criterion(pred, mask)
                    + 0.4 * criterion(aux4, mask_16)
                    + 0.2 * criterion(aux3, mask_32))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        tp, tn, fp, fn = compute_metrics(pred.detach(), mask)
        tr_tp += tp; tr_tn += tn; tr_fp += fp; tr_fn += fn

    # ── VALIDATION ─────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    v_tp = v_tn = v_fp = v_fn = 0

    with torch.no_grad():
        for img1, img2, mask in val_loader:
            img1, img2, mask = img1.to(device), img2.to(device), mask.to(device)
            with torch.amp.autocast('cuda'):
                pred, _, _ = model(img1, img2)
            val_loss += criterion(pred, mask).item()
            tp, tn, fp, fn = compute_metrics(pred, mask)
            v_tp += tp; v_tn += tn; v_fp += fp; v_fn += fn

    # ── METRICS ────────────────────────────────────────────
    tr_m  = finalize_metrics(tr_tp, tr_tn, tr_fp, tr_fn)
    val_m = finalize_metrics(v_tp,  v_tn,  v_fp,  v_fn)

    scheduler.step()
    current_lr_bb   = scheduler.get_last_lr()[0]
    current_lr_head = scheduler.get_last_lr()[1]

    history["train_loss"].append(train_loss / len(train_loader))
    history["val_loss"  ].append(val_loss   / len(val_loader))
    history["train_acc" ].append(tr_m["acc"]);  history["val_acc" ].append(val_m["acc"])
    history["train_prec"].append(tr_m["prec"]); history["val_prec"].append(val_m["prec"])
    history["train_rec" ].append(tr_m["rec"]);  history["val_rec" ].append(val_m["rec"])
    history["train_f1"  ].append(tr_m["f1"]);   history["val_f1"  ].append(val_m["f1"])
    history["train_iou" ].append(tr_m["iou"]);  history["val_iou" ].append(val_m["iou"])

    # ── SAVE BEST ──────────────────────────────────────────
    if val_m["f1"] > best_val_f1:
        best_val_f1      = val_m["f1"]
        best_epoch       = epoch + 1
        no_improve_count = 0
        torch.save(model.state_dict(), "best_model_stage1_fixed.pth")
        saved_marker = " ✓ BEST"
    else:
        no_improve_count += 1
        saved_marker = f"  (no↑ {no_improve_count}/{EARLY_STOP_PATIENCE})"

    # ── PERIODIC CHECKPOINT ────────────────────────────────
    if (epoch + 1) % CHECKPOINT_EVERY == 0:
        ckpt_path = f"ckpt_stage1_ep{epoch+1}.pth"
        torch.save(model.state_dict(), ckpt_path)
        saved_marker += f"  [ckpt saved]"

    # ── PRINT ──────────────────────────────────────────────
    m, s = divmod(int(time.time() - start_time), 60)
    ep_str = f"{epoch+1}/{num_epochs}"  
    print(
        f"{ep_str:7} | {m:2d}m {s:2d}s | "
        f"{history['train_loss'][-1]:7.4f} | "
        f"{tr_m['acc']:6.4f} | {tr_m['prec']:7.4f} | {tr_m['rec']:6.4f} | "
        f"{tr_m['f1']:6.4f} | {tr_m['iou']:7.4f} | "
        f"{history['val_loss'][-1]:7.4f} | "
        f"{val_m['acc']:6.4f} | {val_m['prec']:7.4f} | {val_m['rec']:6.4f} | "
        f"{val_m['f1']:6.4f} | {val_m['iou']:7.4f} | "
        f"{current_lr_bb:.3e} | {current_lr_head:.3e}"
        f"{saved_marker}"
    )

    if (epoch + 1) % 10 == 0:
        elapsed = int(time.time() - training_start)
        em, es  = divmod(elapsed, 60)
        print("─" * 155)
        print(f"  ↳  Best so far: Val F1={best_val_f1:.4f} @ Ep {best_epoch}"
              f"  |  Elapsed: {em}m {es}s"
              f"  |  No-improve streak: {no_improve_count}/{EARLY_STOP_PATIENCE}")
        print("─" * 155)

    # ── EARLY STOP ─────────────────────────────────────────
    if no_improve_count >= EARLY_STOP_PATIENCE:
        print(f"\n  ⛔ Early stopping at epoch {epoch+1} "
              f"— no Val F1 improvement for {EARLY_STOP_PATIENCE} epochs")
        break

# ── FINAL SUMMARY ──────────────────────────────────────────
total_time = int(time.time() - training_start)
tm, ts = divmod(total_time, 60)
th, tm = divmod(tm, 60)
print("\n" + "=" * 155)
print(f"  STAGE 1 COMPLETE")
print(f"  Best Val F1 : {best_val_f1:.4f}  @ Epoch {best_epoch}")
print(f"  Total Time  : {th}h {tm}m {ts}s")
print(f"  Saved       : best_model_stage1_fixed.pth")
print("=" * 155)


  STAGE 1 TRAINING  —  ResNet-50 Multi-Scale Attn  | 50 epochs | Early-stop patience=25
     Ep |   Time |  TrLoss |  TrAcc |  TrPrec |  TrRec |   TrF1 |   TrIoU |  VlLoss |  VlAcc |  VlPrec |  VlRec |   VlF1 |   VlIoU |     BB_LR |     HD_LR
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
1/50    |  8m  2s |  1.7144 | 0.8419 |  0.5941 | 0.6339 | 0.6134 |  0.4423 |  1.4324 | 0.8426 |  0.5774 | 0.8137 | 0.6755 |  0.5100 | 9.902e-06 | 9.892e-05 ✓ BEST
2/50    |  7m 53s |  1.5486 | 0.8619 |  0.6489 | 0.6543 | 0.6516 |  0.4832 |  1.4097 | 0.8626 |  0.6284 | 0.7772 | 0.6949 |  0.5324 | 9.611e-06 | 9.572e-05 ✓ BEST
3/50    |  7m 53s |  1.4801 | 0.8703 |  0.6692 | 0.6796 | 0.6744 |  0.5087 |  1.3923 | 0.8663 |  0.6344 | 0.7936 | 0.7051 |  0.5446 | 9.141e-06 | 9.055e-05 ✓ BEST
4/50    |  7m 53s |  1.4632 | 0.8734 |  0.6775 | 0.6868 | 0.6821 |  0.5176 |  1.3720 | 0.8744 |  0.6675 | 0.7500

In [15]:
model.load_state_dict(torch.load("best_model_stage1_fixed.pth"))
model.eval()

def predict_tta(model, img1, img2):
    """Average predictions over 4 flips — free +0.5–1% F1"""
    preds = []
    flip_pairs = [
        (lambda x: x,          lambda x: x),           # original
        (TF.hflip,              TF.hflip),              # hflip → unflip pred
        (TF.vflip,              TF.vflip),              # vflip → unflip pred
        (lambda x: TF.hflip(TF.vflip(x)),
         lambda x: TF.hflip(TF.vflip(x))),             # both flips
    ]
    with torch.no_grad():
        for flip_in, flip_out in flip_pairs:
            i1 = torch.stack([flip_in(img1[b]) for b in range(img1.shape[0])])
            i2 = torch.stack([flip_in(img2[b]) for b in range(img2.shape[0])])
            with torch.cuda.amp.autocast():
                p, _, _ = model(i1, i2)
            p_sig = torch.sigmoid(p)
            p_unflipped = torch.stack([flip_out(p_sig[b]) for b in range(p_sig.shape[0])])
            preds.append(p_unflipped)
    return torch.stack(preds).mean(0)  # averaged probability map

test_loss = 0.0
t_tp = t_tn = t_fp = t_fn = 0

with torch.no_grad():
    for img1, img2, mask in test_loader:
        img1, img2, mask = img1.to(device), img2.to(device), mask.to(device)
        avg_prob = predict_tta(model, img1, img2)          # [B,1,H,W] 0–1
        pred_bin = (avg_prob > 0.5).float()
        tp = (pred_bin * mask).sum().float()
        tn = ((1-pred_bin)*(1-mask)).sum().float()
        fp = (pred_bin*(1-mask)).sum().float()
        fn = ((1-pred_bin)*mask).sum().float()
        t_tp += tp; t_tn += tn; t_fp += fp; t_fn += fn

test_m = finalize_metrics(t_tp, t_tn, t_fp, t_fn)

print("\n" + "═" * 50)
print("  TEST RESULTS — code1-10 (ResNet-50 + TTA)")
print("═" * 50)
print(f"  Accuracy  : {test_m['acc']:.4f}")
print(f"  Precision : {test_m['prec']:.4f}")
print(f"  Recall    : {test_m['rec']:.4f}")
print(f"  F1 Score  : {test_m['f1']:.4f}")
print(f"  IoU       : {test_m['iou']:.4f}")
print("═" * 50)

/tmp/ipykernel_24/2340180705.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



══════════════════════════════════════════════════
  TEST RESULTS — code1-10 (ResNet-50 + TTA)
══════════════════════════════════════════════════
  Accuracy  : 0.8886
  Precision : 0.7101
  Recall    : 0.7701
  F1 Score  : 0.7389
  IoU       : 0.5859
══════════════════════════════════════════════════
